# 04 — Inferenza dei Layer Normativi e Heatmap di Ibridità

Questo notebook implementa il cuore metodologico del progetto: **i livelli gerarchici
non sono predefiniti ma emergono dai dati** della specifica materia analizzata.

## Pipeline in quattro fasi

| Fase | Input | Operazione | Output |
|---|---|---|---|
| **A** | Segmenti di ogni atto | LLM 1: descrizione funzionale contestualizzata per segmento | `segments_descriptions.csv` |
| **B** | Descrizioni funzionali | Embedding + UMAP + HDBSCAN → cluster = layer emergenti | embedding in memoria |
| **B.2** | 10 descrizioni representative per cluster | LLM 2: assegna nome al layer | `layer_mapping.csv` |
| **C** | Articoli + layer noti | LLM 3: distribuzione % articolo × layer | `nodes_heatmap.csv` |
| **D** | Matrice per atto | Entropia media → score di ibridità continuo | `nodes_hybridity.csv` |

## Principio metodologico

Il clustering avviene a livello di **segmento** (articolo o considerando), non di atto.
Questo permette di trovare i livelli gerarchici reali presenti in quella materia,
indipendentemente da come sono distribuiti tra gli atti.

Un atto è **ibrido** se i suoi articoli hanno distribuzioni molto diverse tra loro
— alcuni concentrati su livelli alti della gerarchia, altri su livelli di dettaglio
tecnico. L'ibridità è misurabile come **entropia media delle distribuzioni** per articolo.

## Output principali

| File | Contenuto |
|---|---|
| `segments_descriptions.csv` | Una riga per segmento con la descrizione funzionale prodotta da LLM 1 |
| `layer_mapping.csv` | Mappatura cluster → nome layer con descrizione e descrizioni representative |
| `nodes_heatmap.csv` | Una riga per (celex, articolo) con % per ogni layer |
| `nodes_hybridity.csv` | Una riga per atto con score ibridità, layer dominante, n_articoli |

---

> **Nota sul costo API**: questa fase chiama l'LLM una volta per segmento (Fase A)
> e una volta per articolo (Fase C). Con una rete di ~200 atti e ~20 segmenti/atto
> il totale è dell'ordine di 4.000-6.000 chiamate. Il checkpointing granulare
> permette di riprendere in caso di interruzione.

## 0. Setup e Parametri

In [ ]:
import os
import sys
import json
import time
import math
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter, defaultdict

from openai import OpenAI
import openai

sys.path.append('..')
from config_golden_power import (
    MATERIA_NAME, NODES_TEXTS, LAYER_MAPPING, NODES_LAYERS, FUNC_CHECKPOINT
)

# ── Percorsi ────────────────────────────────────────────────────────────────
output_path = os.path.join('..', 'data', 'output', MATERIA_NAME)
Path(output_path).mkdir(parents=True, exist_ok=True)

INPUT_FILE              = os.path.join(output_path, NODES_TEXTS)
SEGMENTS_DESC_FILE      = os.path.join(output_path, 'segments_descriptions.csv')
LAYER_MAPPING_FILE      = os.path.join(output_path, LAYER_MAPPING)
NODES_HEATMAP_FILE      = os.path.join(output_path, 'nodes_heatmap.csv')
NODES_HYBRIDITY_FILE    = os.path.join(output_path, 'nodes_hybridity.csv')
HEATMAP_CHECKPOINT_FILE = os.path.join(output_path, 'heatmap_checkpoint.csv')

# ── Parametri LLM ───────────────────────────────────────────────────────────
LLM_MODEL           = 'gpt-4o'   # modello OpenAI usato per tutte e tre le invocazioni
LLM_MAX_TOKENS      = 300    # Fase A: descrizioni brevi
LLM_MAX_TOKENS_PCT  = 500    # Fase C: JSON con percentuali
LLM_DELAY_SECONDS   = 0.3    # pausa tra chiamate API
LLM_MAX_RETRIES     = 3      # tentativi in caso di errore
LLM_RETRY_DELAY     = 5.0    # secondi tra retry

# ── Parametri checkpoint ────────────────────────────────────────────────────
CHECKPOINT_EVERY    = 100    # segmenti tra un salvataggio e il successivo

# ── Parametri clustering ────────────────────────────────────────────────────
EMBEDDING_MODEL     = 'all-mpnet-base-v2'
UMAP_N_COMPONENTS   = 10     # dim ridotta per HDBSCAN
UMAP_N_NEIGHBORS    = 15
UMAP_MIN_DIST       = 0.0    # 0.0 ottimizza per clustering
HDBSCAN_MIN_CLUSTER = 5      # min segmenti per formare un layer
HDBSCAN_MIN_SAMPLES = 3
N_REPR_DESCRIPTIONS = 10     # descrizioni representative per naming layer

# ── Parametri ibridità ───────────────────────────────────────────────────────
# Segmenti inclusi nel calcolo dell'ibridità
# Solo gli articoli — i considerando hanno funzione diversa (giustificativa, non
# prescrittiva) e la loro varianza riflette struttura retorica, non patologia.
HYBRIDITY_SEGMENT_TYPES = ['articolo']   # valori possibili: 'articolo', 'considerando', 'allegato'

print(f"Input:              {INPUT_FILE}")
print(f"Segments desc:      {SEGMENTS_DESC_FILE}")
print(f"Layer mapping:      {LAYER_MAPPING_FILE}")
print(f"Heatmap:            {NODES_HEATMAP_FILE}")
print(f"Hybridity:          {NODES_HYBRIDITY_FILE}")
print(f"Modello LLM:        {LLM_MODEL}")
print(f"Modello embedding:  {EMBEDDING_MODEL}")

## 1. Caricamento Dati e Verifica Input

In [ ]:
nodes = pd.read_csv(INPUT_FILE)
print(f"Nodi totali: {len(nodes)}")
print(f"Colonne: {list(nodes.columns)}")
print()

# Verifica colonne obbligatorie
REQUIRED_COLS = ['Id', 'Label', 'segments', 'text_status', 'title']
missing_cols = [c for c in REQUIRED_COLS if c not in nodes.columns]
if missing_cols:
    raise RuntimeError(
        f"Colonne mancanti in {INPUT_FILE}: {missing_cols}.\n"
        "Assicurarsi che il notebook 03_enrich_texts.ipynb sia stato eseguito completamente."
    )

# Seleziona solo atti con testo estratto con successo
nodes_ok = nodes[nodes['text_status'] == 'ok'].copy()
nodes_fail = nodes[nodes['text_status'] != 'ok'].copy()

print(f"Atti con testo (text_status=ok): {len(nodes_ok)}")
print(f"Atti senza testo (esclusi):      {len(nodes_fail)}")
print()
print("Distribuzione text_status:")
print(nodes['text_status'].value_counts().to_string())

## 2. Esplosione dei Segmenti

Ogni atto contiene una colonna `segments` con una lista JSON di segmenti strutturati
(articoli, considerando, allegati). Questa cella costruisce un DataFrame flat
`segments_df` con una riga per segmento — l'unità di analisi per il clustering.

In [ ]:
def parse_segments(row):
    """
    Parsa la colonna 'segments' di una riga e restituisce una lista di dict
    arricchiti con celex e titolo dell'atto.
    """
    celex = row.get('Label', row['Id'])
    title = str(row.get('title', ''))
    raw   = row.get('segments')

    if pd.isna(raw) or not str(raw).strip():
        return []

    try:
        segs = json.loads(str(raw))
    except (json.JSONDecodeError, ValueError):
        return []

    result = []
    for s in segs:
        result.append({
            'celex':         celex,
            'node_id':       row['Id'],
            'title_atto':    title,
            'tipo':          s.get('tipo', ''),
            'identificatore': s.get('identificatore', ''),
            'testo':         s.get('testo', ''),
        })
    return result


all_segments = []
for _, row in nodes_ok.iterrows():
    all_segments.extend(parse_segments(row))

segments_df = pd.DataFrame(all_segments)

# Assegna un ID univoco per segmento
segments_df['segment_id'] = (
    segments_df['celex'] + '__' +
    segments_df['tipo'] + '__' +
    segments_df['identificatore'].astype(str)
)

# Rimuove segmenti con testo vuoto o troppo corto per essere informativi
MIN_TESTO_LEN = 30
before = len(segments_df)
segments_df = segments_df[segments_df['testo'].str.len() >= MIN_TESTO_LEN].copy()
segments_df = segments_df.reset_index(drop=True)

print(f"Segmenti totali estratti:    {before:,}")
print(f"Segmenti con testo ≥{MIN_TESTO_LEN} car: {len(segments_df):,}")
print(f"Segmenti scartati (troppo brevi): {before - len(segments_df):,}")
print()
print("Distribuzione per tipo:")
print(segments_df['tipo'].value_counts().to_string())
print()
print("Segmenti medi per atto:")
print(f"  {segments_df.groupby('celex').size().describe()[['mean','50%','max']].to_string()}")

## 3. Fase A — Descrizione Funzionale per Segmento (LLM 1)

Per ogni segmento (articolo, considerando, allegato) l'LLM produce una **descrizione
funzionale contestualizzata alla materia**: cosa fa normativamente quel segmento
in questo specifico contesto normativo.

**Questo è il passaggio che rende il metodo specifico per materia**: lo stesso articolo
"Definizioni" viene descritto diversamente se appare in un regolamento sugli FDI
rispetto a uno sulla privacy.

> **Checkpoint**: i risultati vengono salvati in `segments_descriptions.csv`
> dopo ogni `CHECKPOINT_EVERY` segmenti. Se la cella viene interrotta,
> rieseguirla riparte dal punto di interruzione.

In [ ]:
TEMA_DESCRIZIONE = (
    "controllo degli investimenti diretti esteri (FDI Screening) "
    "e poteri speciali dello Stato (Golden Power) sulla sicurezza "
    "nazionale e le infrastrutture strategiche"
)

def build_prompt_functional_description(testo: str, tipo: str, identificatore: str,
                                         title_atto: str, tema: str) -> str:
    return f"""Sei un esperto di diritto europeo specializzato nella materia: {tema}.

Leggi questo segmento di un atto normativo e descrivi in 1-2 frasi cosa fa
normativamente — quale funzione specifica svolge in questa materia.

Sii preciso sulla funzione. Esempi di descrizioni corrette:
- "Stabilisce i principi fondamentali che giustificano l'intervento statale negli investimenti esteri per ragioni di sicurezza nazionale."
- "Definisce la procedura di notifica preventiva che gli investitori esteri devono seguire prima di acquisire partecipazioni in settori strategici."
- "Specifica le soglie percentuali di controllo societario oltre le quali scatta l'obbligo di screening da parte delle autorità nazionali."
- "Elenca i settori economici considerati critici ai fini dell'applicazione dei poteri di intervento statale."

Titolo atto: {title_atto}
Segmento ({tipo} {identificatore}):
{testo}

Rispondi SOLO con la descrizione funzionale (1-2 frasi), nessun altro testo."""


def call_llm_with_retry(client: OpenAI, prompt: str,
                         max_tokens: int = LLM_MAX_TOKENS,
                         max_retries: int = LLM_MAX_RETRIES) -> tuple[str, str]:
    """
    Chiama l'LLM OpenAI con retry automatico su errori transitori.

    Restituisce (testo_risposta, status) dove status è 'ok' | 'error' | 'empty'.
    """
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=LLM_MODEL,
                max_tokens=max_tokens,
                messages=[{'role': 'user', 'content': prompt}]
            )
            text = response.choices[0].message.content.strip()
            if not text:
                return '', 'empty'
            return text, 'ok'

        except openai.RateLimitError:
            wait = LLM_RETRY_DELAY * (attempt + 1) * 2
            print(f"  [RateLimit] attesa {wait:.0f}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)

        except openai.APIStatusError as e:
            if attempt < max_retries - 1:
                time.sleep(LLM_RETRY_DELAY)
            else:
                return f'ERROR: {e}', 'error'

        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(LLM_RETRY_DELAY)
            else:
                return f'ERROR: {e}', 'error'

    return 'ERROR: max retries exceeded', 'error'


print("Funzioni di Fase A definite.")
print(f"Tema: {TEMA_DESCRIZIONE}")

In [ ]:
# ── Gestione checkpoint ──────────────────────────────────────────────────────
if os.path.exists(SEGMENTS_DESC_FILE):
    segs_done = pd.read_csv(SEGMENTS_DESC_FILE)
    done_ids  = set(segs_done['segment_id'])
    print(f"Checkpoint trovato: {len(done_ids):,} segmenti già descritti.")
else:
    segs_done = pd.DataFrame()
    done_ids  = set()
    print("Nessun checkpoint, si parte da zero.")

segments_todo = segments_df[~segments_df['segment_id'].isin(done_ids)].copy()
print(f"Segmenti da descrivere: {len(segments_todo):,}")

if len(segments_todo) == 0:
    print("✓ Tutti i segmenti già descritti — si passa alla Fase B.")

In [ ]:
%%time
client = OpenAI()  # usa OPENAI_API_KEY dall'environment

new_rows      = []
n_ok          = 0
n_error       = 0
total         = len(segments_todo)

for i, (_, seg) in enumerate(segments_todo.iterrows()):

    prompt = build_prompt_functional_description(
        testo        = seg['testo'],
        tipo         = seg['tipo'],
        identificatore = seg['identificatore'],
        title_atto   = seg['title_atto'],
        tema         = TEMA_DESCRIZIONE,
    )

    descrizione, status = call_llm_with_retry(client, prompt)

    new_rows.append({
        'segment_id':           seg['segment_id'],
        'celex':                seg['celex'],
        'node_id':              seg['node_id'],
        'tipo':                 seg['tipo'],
        'identificatore':       seg['identificatore'],
        'testo_originale':      seg['testo'],
        'descrizione_funzionale': descrizione,
        'llm_status':           status,
    })

    if status == 'ok':
        n_ok += 1
    else:
        n_error += 1

    # Checkpoint
    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == total:
        batch_df  = pd.DataFrame(new_rows)
        combined  = pd.concat([segs_done, batch_df], ignore_index=True) \
                      if not segs_done.empty else batch_df
        combined.to_csv(SEGMENTS_DESC_FILE, index=False)
        pct = (i + 1) / total * 100
        print(f"  [{i+1:>5}/{total}] {pct:5.1f}%  ok: {n_ok}  errori: {n_error}")

    time.sleep(LLM_DELAY_SECONDS)

print()
print("=" * 50)
print("FASE A COMPLETATA")
print("=" * 50)
print(f"  Descrizioni prodotte: {n_ok:,}")
print(f"  Errori:               {n_error:,}")

## 4. Fase B — Embedding e Clustering → Layer Emergenti

Tutte le descrizioni funzionali vengono embeddate con `all-mpnet-base-v2` e poi
ridotte con UMAP + clusterizzate con HDBSCAN.

Ogni cluster che emerge rappresenta un **livello gerarchico specifico per questa materia**.
HDBSCAN decide quanti cluster ci sono — non lo decide l'analista.

> I segmenti assegnati al cluster `-1` (noise) vengono conservati nel DataFrame
> ma esclusi dal naming e dal calcolo dell'ibridità.

In [ ]:
from sentence_transformers import SentenceTransformer
import umap
import hdbscan

# Carica descriptions dal checkpoint finale
segs_desc = pd.read_csv(SEGMENTS_DESC_FILE)

# Filtra solo descrizioni valide
segs_valid = segs_desc[
    (segs_desc['llm_status'] == 'ok') &
    segs_desc['descrizione_funzionale'].notna() &
    (segs_desc['descrizione_funzionale'].str.len() > 10)
].copy()

print(f"Segmenti con descrizione valida: {len(segs_valid):,} / {len(segs_desc):,}")
print()

# ── Embedding ────────────────────────────────────────────────────────────────
print(f"Carico modello embedding: {EMBEDDING_MODEL} ...")
encoder = SentenceTransformer(EMBEDDING_MODEL)

print("Calcolo embeddings...")
texts_to_embed = segs_valid['descrizione_funzionale'].tolist()

embeddings = encoder.encode(
    texts_to_embed,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# ── UMAP — riduzione dimensionale ────────────────────────────────────────────
print(f"UMAP: {embeddings.shape[1]}d → {UMAP_N_COMPONENTS}d ...")

reducer = umap.UMAP(
    n_components  = UMAP_N_COMPONENTS,
    n_neighbors   = UMAP_N_NEIGHBORS,
    min_dist      = UMAP_MIN_DIST,
    metric        = 'cosine',
    random_state  = 42,
    low_memory    = False,
)
embeddings_reduced = reducer.fit_transform(embeddings)
print(f"Shape ridotta: {embeddings_reduced.shape}")

In [ ]:
# ── HDBSCAN — clustering ─────────────────────────────────────────────────────
print(f"HDBSCAN (min_cluster_size={HDBSCAN_MIN_CLUSTER}, min_samples={HDBSCAN_MIN_SAMPLES}) ...")

clusterer = hdbscan.HDBSCAN(
    min_cluster_size  = HDBSCAN_MIN_CLUSTER,
    min_samples       = HDBSCAN_MIN_SAMPLES,
    cluster_selection_method = 'eom',
    prediction_data   = True,
)
cluster_labels = clusterer.fit_predict(embeddings_reduced)

segs_valid = segs_valid.copy()
segs_valid['cluster_id'] = cluster_labels

# Statistiche clustering
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise    = (cluster_labels == -1).sum()
pct_noise  = n_noise / len(cluster_labels) * 100

print()
print("=" * 50)
print("RISULTATO CLUSTERING")
print("=" * 50)
print(f"  Cluster trovati (layer):  {n_clusters}")
print(f"  Segmenti noise (-1):      {n_noise:,}  ({pct_noise:.1f}%)")
print()
print("Distribuzione segmenti per cluster:")
cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
for cid, cnt in cluster_counts.items():
    label = 'NOISE' if cid == -1 else f'cluster_{cid}'
    bar   = '█' * min(40, int(cnt / max(cluster_counts) * 40))
    print(f"  {label:>12}: {cnt:>5}  {bar}")

## 5. Fase B.2 — Ordinamento Gerarchico e Naming dei Layer (LLM 2)

Due operazioni:

1. **Ordinamento gerarchico**: i cluster vengono ordinati per posizione nella gerarchia
   usando la struttura della rete di citazioni. Gli atti più spesso citati (alto indegree)
   tendono a contenere segmenti di livello apicale (principi, obiettivi) mentre gli atti
   con alto outdegree tendono a contenere segmenti tecnici di dettaglio.

2. **Naming**: l'LLM riceve le 10 descrizioni funzionali più rappresentative di ogni
   cluster e assegna un nome descrittivo al layer.

In [ ]:
from config_golden_power import EDGES_FOCAL

EDGES_FILE = os.path.join(output_path, EDGES_FOCAL)

try:
    edges = pd.read_csv(EDGES_FILE)
    print(f"Archi caricati: {len(edges):,}")

    # Calcola indegree per ogni celex (proxy di 'quanto è citato' → apicale)
    # In edges, ':END_ID' è il nodo citato (la destinazione della citazione)
    # Un atto molto citato è probabilmente fondativo → livelli apicali
    indegree = edges[':END_ID'].value_counts().rename('indegree')

    # Per ogni cluster calcola l'indegree medio dei propri segmenti
    # (proxy di gerarchia)
    segs_with_degree = segs_valid.merge(
        indegree.reset_index().rename(columns={':END_ID': 'node_id_edge', 'indegree': 'indegree'}),
        left_on='node_id', right_on='node_id_edge', how='left'
    )
    segs_with_degree['indegree'] = segs_with_degree['indegree'].fillna(0)

    cluster_hierarchy = (
        segs_with_degree[segs_with_degree['cluster_id'] != -1]
        .groupby('cluster_id')['indegree']
        .mean()
        .sort_values(ascending=False)  # indegree alto → livello apicale → rank basso
    )
    cluster_rank = {cid: rank for rank, cid in enumerate(cluster_hierarchy.index, start=1)}
    print("Ordinamento gerarchico (indegree medio per cluster):")
    for cid, mean_ind in cluster_hierarchy.items():
        print(f"  cluster_{cid} → rank {cluster_rank[cid]} (indegree medio: {mean_ind:.2f})")

except FileNotFoundError:
    print(f"⚠️  {EDGES_FILE} non trovato — ordinamento gerarchico saltato.")
    print("I cluster verranno ordinati per dimensione (cluster più grande = rank 1).")
    non_noise = cluster_counts[cluster_counts.index != -1].sort_values(ascending=False)
    cluster_rank = {cid: rank for rank, cid in enumerate(non_noise.index, start=1)}

print(f"\nRanking finale: {cluster_rank}")

In [ ]:
def get_representative_descriptions(segs_df: pd.DataFrame, cluster_id: int,
                                     n: int = N_REPR_DESCRIPTIONS) -> list[str]:
    """
    Seleziona le N descrizioni funzionali più rappresentative di un cluster.
    Usa la probabilità di appartenenza HDBSCAN se disponibile, altrimenti campione casuale.
    """
    mask  = segs_df['cluster_id'] == cluster_id
    subset = segs_df[mask].copy()

    if 'cluster_prob' in subset.columns:
        subset = subset.nlargest(n, 'cluster_prob')
    else:
        subset = subset.sample(min(n, len(subset)), random_state=42)

    return subset['descrizione_funzionale'].tolist()


def build_prompt_layer_naming(descriptions: list[str], tema: str,
                               rank: int, n_total_layers: int) -> str:
    desc_list = '\n'.join([f"  {i+1}. {d}" for i, d in enumerate(descriptions)])
    return f"""Sei un esperto di diritto europeo specializzato nella materia: {tema}.

Ho condotto un'analisi clustering su un corpus di atti normativi e ho trovato
{n_total_layers} livelli gerarchici distinti presenti in questa materia.
Questo è il livello {rank} su {n_total_layers} in ordine dalla norma più apicale
alla più tecnica di dettaglio.

Ecco le descrizioni funzionali dei segmenti più rappresentativi di questo livello:

{desc_list}

Basandoti su queste descrizioni, assegna un nome preciso a questo livello gerarchico
e scrivi una breve descrizione (2-3 frasi) di cosa caratterizza le norme che vi appartengono.

Rispondi SOLO in questo formato JSON (nessun altro testo, nessun markdown):
{{"nome": "Nome del Layer", "descrizione": "Descrizione in 2-3 frasi."}}"""


# Aggiunge probabilità cluster se disponibile
if hasattr(clusterer, 'probabilities_'):
    segs_valid['cluster_prob'] = clusterer.probabilities_

# ── Loop naming ──────────────────────────────────────────────────────────────
layer_records   = []
n_valid_clusters = len([cid for cid in set(cluster_labels) if cid != -1])

for cluster_id in sorted(cluster_rank.keys()):
    rank = cluster_rank[cluster_id]
    repr_descs = get_representative_descriptions(segs_valid, cluster_id)

    prompt = build_prompt_layer_naming(
        descriptions   = repr_descs,
        tema           = TEMA_DESCRIZIONE,
        rank           = rank,
        n_total_layers = n_valid_clusters,
    )

    response_text, status = call_llm_with_retry(
        client, prompt, max_tokens=400
    )

    nome        = f'Layer_{rank}'
    descrizione = ''

    if status == 'ok':
        try:
            # Pulisce eventuali backtick markdown
            clean = response_text.replace('```json', '').replace('```', '').strip()
            parsed = json.loads(clean)
            nome        = parsed.get('nome', nome)
            descrizione = parsed.get('descrizione', '')
        except json.JSONDecodeError:
            # Fallback: usa la risposta raw come nome
            nome = response_text[:80].strip()
            print(f"  ⚠️  cluster_{cluster_id}: parsing JSON fallito, uso raw text come nome")

    layer_records.append({
        'cluster_id':             cluster_id,
        'layer_rank':             rank,
        'layer_name':             nome,
        'layer_description':      descrizione,
        'n_segments':             (segs_valid['cluster_id'] == cluster_id).sum(),
        'repr_descriptions':      json.dumps(repr_descs, ensure_ascii=False),
        'llm_status':             status,
    })

    print(f"  Rank {rank:>2} | cluster_{cluster_id:>3} → '{nome}'")
    time.sleep(LLM_DELAY_SECONDS)

layer_mapping_df = pd.DataFrame(layer_records).sort_values('layer_rank')
layer_mapping_df.to_csv(LAYER_MAPPING_FILE, index=False)

print()
print("=" * 50)
print("LAYER TROVATI")
print("=" * 50)
for _, row in layer_mapping_df.iterrows():
    print(f"  [{row['layer_rank']}] {row['layer_name']}")
    print(f"      {row['layer_description'][:120]}...")
    print(f"      N segmenti: {row['n_segments']:,}")
    print()

## 6. Fase C — Distribuzione Percentuale per Articolo (LLM 3)

Con i layer in mano, per ogni **articolo** (non considerando, non allegato) di ogni atto
l'LLM produce una distribuzione percentuale del contenuto tra i layer emersi.

Il risultato per ogni atto è una matrice: **articoli × layer, valori = percentuali**.
Questa è la heatmap visualizzabile nell'applicazione.

> **Perché solo gli articoli?** I considerando hanno funzione giustificativa e retorica —
> spesso coprono intenzionalmente più livelli per costruire l'argomentazione. Gli articoli
> invece hanno funzione prescrittiva e la loro variazione di livello è il segnale di patologia.

In [ ]:
# Carica layer mapping
layer_mapping_df = pd.read_csv(LAYER_MAPPING_FILE)

# Prepara la lista ordinata di layer per i prompt
layer_list = [
    {
        'rank':        int(row['layer_rank']),
        'name':        row['layer_name'],
        'description': row['layer_description'],
    }
    for _, row in layer_mapping_df.sort_values('layer_rank').iterrows()
]

# Dizionario per accesso rapido
layer_names = [l['name'] for l in layer_list]
n_layers    = len(layer_names)

print(f"Layer per la heatmap ({n_layers} totali):")
for l in layer_list:
    print(f"  [{l['rank']}] {l['name']}")

# Seleziona solo gli articoli
articles_df = segments_df[segments_df['tipo'] == 'articolo'].copy()
print(f"\nArticoli totali da classificare: {len(articles_df):,}")
print(f"Atti coinvolti: {articles_df['celex'].nunique():,}")

In [ ]:
def build_prompt_percentage_distribution(testo: str, identificatore: str,
                                          layer_list: list[dict], tema: str) -> str:
    layers_desc = '\n'.join([
        f"  {l['rank']}. {l['name']}: {l['description']}"
        for l in layer_list
    ])
    layer_keys = ', '.join([f'"{l["name"]}"' for l in layer_list])

    return f"""Sei un esperto di diritto europeo specializzato nella materia: {tema}.

Leggi questo articolo e distribuisci il suo contenuto in percentuale tra i seguenti
livelli normativi emersi dall'analisi di questa materia specifica:

{layers_desc}

Regole:
- Le percentuali devono sommare esattamente a 100.
- Assegna 0 ai livelli non presenti nell'articolo.
- Un articolo può appartenere a uno solo (100%) o a più livelli contemporaneamente.
- Sii preciso: se un articolo mescola principi generali (20%) con procedure dettagliate (80%), rifletti questa proporzione.

Articolo {identificatore}:
{testo}

Rispondi SOLO con JSON valido (nessun altro testo, nessun markdown):
{{{layer_keys}: <percentuale intera>}}"""


def parse_percentage_response(response_text: str, layer_names: list[str]) -> dict | None:
    """
    Parsa la risposta JSON dell'LLM e la normalizza a somma 100.
    Restituisce None se il parsing fallisce.
    """
    try:
        clean = response_text.replace('```json', '').replace('```', '').strip()
        parsed = json.loads(clean)
    except json.JSONDecodeError:
        return None

    # Estrae solo i valori per i layer attesi, imposta 0 per quelli mancanti
    values = {name: float(parsed.get(name, 0)) for name in layer_names}

    # Normalizza a 100 se la somma è non-zero
    total = sum(values.values())
    if total <= 0:
        return None
    if abs(total - 100) > 5:  # tolleranza 5%
        values = {k: v / total * 100 for k, v in values.items()}

    return values


print("Funzioni di Fase C definite.")

In [ ]:
%%time
# ── Gestione checkpoint heatmap ──────────────────────────────────────────────
if os.path.exists(HEATMAP_CHECKPOINT_FILE):
    heatmap_done = pd.read_csv(HEATMAP_CHECKPOINT_FILE)
    done_seg_ids = set(heatmap_done['segment_id'])
    print(f"Checkpoint trovato: {len(done_seg_ids):,} articoli già classificati.")
else:
    heatmap_done = pd.DataFrame()
    done_seg_ids = set()
    print("Nessun checkpoint heatmap, si parte da zero.")

articles_todo = articles_df[
    ~articles_df['segment_id'].isin(done_seg_ids)
].copy()
print(f"Articoli da classificare: {len(articles_todo):,}")

# ── Loop classificazione ─────────────────────────────────────────────────────
new_heatmap_rows = []
n_ok_c    = 0
n_error_c = 0
total_c   = len(articles_todo)

for i, (_, art) in enumerate(articles_todo.iterrows()):

    prompt = build_prompt_percentage_distribution(
        testo          = art['testo'],
        identificatore = art['identificatore'],
        layer_list     = layer_list,
        tema           = TEMA_DESCRIZIONE,
    )

    response_text, status = call_llm_with_retry(
        client, prompt, max_tokens=LLM_MAX_TOKENS_PCT
    )

    distribution = None
    if status == 'ok':
        distribution = parse_percentage_response(response_text, layer_names)
        if distribution is None:
            status = 'parse_error'

    row = {
        'segment_id':    art['segment_id'],
        'celex':         art['celex'],
        'node_id':       art['node_id'],
        'articolo_id':   art['identificatore'],
        'llm_status':    status,
    }

    if distribution:
        for layer_name, pct in distribution.items():
            # Colonna-safe: sostituisce spazi e caratteri speciali
            col = 'pct__' + layer_name.replace(' ', '_').replace('/', '_')[:50]
            row[col] = round(pct, 2)
        n_ok_c += 1
    else:
        # Imposta tutto a 0 se il parsing è fallito
        for layer_name in layer_names:
            col = 'pct__' + layer_name.replace(' ', '_').replace('/', '_')[:50]
            row[col] = 0.0
        n_error_c += 1

    new_heatmap_rows.append(row)

    # Checkpoint
    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == total_c:
        batch_df = pd.DataFrame(new_heatmap_rows)
        combined = pd.concat([heatmap_done, batch_df], ignore_index=True) \
                     if not heatmap_done.empty else batch_df
        combined.to_csv(HEATMAP_CHECKPOINT_FILE, index=False)
        pct_done = (i + 1) / total_c * 100
        print(f"  [{i+1:>5}/{total_c}] {pct_done:5.1f}%  ok: {n_ok_c}  errori: {n_error_c}")

    time.sleep(LLM_DELAY_SECONDS)

print()
print("=" * 50)
print("FASE C COMPLETATA")
print("=" * 50)
print(f"  Articoli classificati: {n_ok_c:,}")
print(f"  Errori:                {n_error_c:,}")

In [ ]:
# Carica checkpoint finale e salva nodes_heatmap.csv
heatmap_final = pd.read_csv(HEATMAP_CHECKPOINT_FILE)

# Aggiunge i nomi originali dei layer come metadato per leggibilità
# (i nomi delle colonne pct__ sono safe per CSV, questo dizionario è per la UI)
pct_cols = [c for c in heatmap_final.columns if c.startswith('pct__')]
col_to_layer = {
    'pct__' + name.replace(' ', '_').replace('/', '_')[:50]: name
    for name in layer_names
}

heatmap_final.to_csv(NODES_HEATMAP_FILE, index=False)
print(f"Salvato: {NODES_HEATMAP_FILE}")
print(f"Righe: {len(heatmap_final):,}")
print(f"Colonne percentuale: {pct_cols}")

# Verifica qualità: distribuzione media per layer
ok_mask = heatmap_final['llm_status'] == 'ok'
print()
print("Distribuzione media % per layer (articoli classificati correttamente):")
for col in pct_cols:
    mean_pct = heatmap_final.loc[ok_mask, col].mean()
    layer_orig = col_to_layer.get(col, col)
    bar = '█' * int(mean_pct / 2)
    print(f"  {layer_orig[:45]:.<46} {mean_pct:5.1f}%  {bar}")

## 7. Fase D — Score di Ibridità per Atto

Lo score di ibridità misura quanto gli articoli di un atto variano nel loro livello
gerarchico. Un atto **puro** ha tutti gli articoli concentrati sullo stesso layer.
Un atto **ibrido** ha articoli che spaziano su layer molto diversi.

**Metrica scelta: entropia media degli articoli**

Per ogni articolo calcola l'entropia di Shannon della sua distribuzione:
$$H(a) = -\sum_{l} p_{al} \log_2(p_{al} + \epsilon)$$

Lo score dell'atto è la media delle entropie dei propri articoli:
$$\text{hybridity}(A) = \frac{1}{|\text{articoli}|} \sum_{a \in A} H(a)$$

**Interpretazione**: un atto con score vicino a 0 ha tutti gli articoli monofunzionali
(ogni articolo fa una cosa sola a un livello preciso). Un atto con score alto ha
articoli internamente misti, segnale di complessità patologica.

In [ ]:
def entropy_distribution(row: pd.Series, pct_cols: list[str]) -> float:
    """
    Calcola l'entropia di Shannon normalizzata (0-1) della distribuzione
    di un articolo sui layer.

    Normalizzata rispetto al massimo teorico log2(n_layers) per rendere
    il valore comparabile tra materie con numero di layer diverso.
    """
    eps = 1e-9
    probs = np.array([float(row.get(col, 0)) for col in pct_cols]) / 100.0
    probs = np.clip(probs, 0, 1)

    if probs.sum() < eps:
        return 0.0

    # Normalizza per sicurezza
    probs = probs / probs.sum()

    raw_entropy = -np.sum(probs * np.log2(probs + eps))
    max_entropy = math.log2(len(pct_cols)) if len(pct_cols) > 1 else 1.0

    return float(raw_entropy / max_entropy)  # 0 = puro, 1 = uniforme


def dominant_layer(row: pd.Series, pct_cols: list[str],
                   col_to_layer: dict) -> str:
    """Restituisce il nome del layer con percentuale più alta."""
    max_col = max(pct_cols, key=lambda c: float(row.get(c, 0)))
    return col_to_layer.get(max_col, max_col)


# Filtra solo articoli con classificazione riuscita
heatmap_ok = heatmap_final[heatmap_final['llm_status'] == 'ok'].copy()
heatmap_ok['entropy'] = heatmap_ok.apply(
    lambda r: entropy_distribution(r, pct_cols), axis=1
)
heatmap_ok['dominant_layer'] = heatmap_ok.apply(
    lambda r: dominant_layer(r, pct_cols, col_to_layer), axis=1
)

# ── Aggregazione per atto ────────────────────────────────────────────────────
def agg_hybridity(group):
    return pd.Series({
        'hybridity_score':     group['entropy'].mean(),
        'hybridity_std':       group['entropy'].std(),
        'hybridity_max':       group['entropy'].max(),
        'n_articles':          len(group),
        'dominant_layer':      group['dominant_layer'].mode()[0]
                               if len(group) > 0 else '',
        'dominant_layer_pct':  (
            group['dominant_layer'].value_counts().iloc[0] / len(group) * 100
            if len(group) > 0 else 0.0
        ),
        # Articoli più ibridi (segnalati all'applicazione per la vista dettaglio)
        'most_hybrid_article': (
            group.nlargest(1, 'entropy')['articolo_id'].iloc[0]
            if len(group) > 0 else ''
        ),
    })

hybridity_df = heatmap_ok.groupby('celex').apply(agg_hybridity).reset_index()

# Aggiunge metadati dall'input originale
meta_cols = ['Id', 'Label', 'title', 'LegalType', 'Year', 'pipeline_level']
meta_cols_available = [c for c in meta_cols if c in nodes.columns]
nodes_meta = nodes[meta_cols_available].copy()
nodes_meta['celex'] = nodes_meta.get('Label', nodes_meta.get('Id'))

hybridity_df = hybridity_df.merge(nodes_meta, on='celex', how='left')

# Ordina per score decrescente (più patologici prima)
hybridity_df = hybridity_df.sort_values('hybridity_score', ascending=False)

hybridity_df.to_csv(NODES_HYBRIDITY_FILE, index=False)

print(f"Salvato: {NODES_HYBRIDITY_FILE}")
print(f"Atti analizzati: {len(hybridity_df):,}")
print()
print("Statistiche score di ibridità:")
desc = hybridity_df['hybridity_score'].describe()
print(f"  Media:    {desc['mean']:.4f}")
print(f"  Mediana:  {desc['50%']:.4f}")
print(f"  Max:      {desc['max']:.4f}")
print(f"  Min:      {desc['min']:.4f}")
print(f"  Std dev:  {desc['std']:.4f}")

## 8. Diagnostica e Verifica Qualità

In [ ]:
print("=" * 60)
print("RIEPILOGO LAYER EMERSI")
print("=" * 60)

for _, row in layer_mapping_df.iterrows():
    n_segs  = row['n_segments']
    pct_seg = n_segs / len(segs_valid) * 100
    bar     = '█' * int(pct_seg)
    print(f"[{row['layer_rank']:>2}] {row['layer_name']}")
    print(f"     Segmenti: {n_segs:,}  ({pct_seg:.1f}%)  {bar}")
    print(f"     {row['layer_description'][:110]}")
    print()

In [ ]:
print("=" * 60)
print("TOP 10 ATTI PIÙ IBRIDI")
print("=" * 60)
print()

top10 = hybridity_df.head(10)
for i, row in top10.iterrows():
    celex_label = row.get('Label', row.get('celex', ''))
    title_short = str(row.get('title', ''))[:70]
    print(f"{row['hybridity_score']:.4f}  {celex_label}")
    print(f"  Layer dominante: {row['dominant_layer']} ({row['dominant_layer_pct']:.0f}% articoli)")
    print(f"  Articoli: {row['n_articles']:>3}  | Articolo più ibrido: {row['most_hybrid_article']}")
    print(f"  {title_short}")
    print()

In [ ]:
print("=" * 60)
print("ANTEPRIMA HEATMAP — ATTO PIÙ IBRIDO")
print("=" * 60)

# Prende l'atto con score più alto e mostra la sua matrice articoli × layer
top_celex  = hybridity_df.iloc[0]['celex']
top_title  = str(hybridity_df.iloc[0].get('title', ''))[:80]
top_score  = hybridity_df.iloc[0]['hybridity_score']

print(f"CELEX: {top_celex}")
print(f"Titolo: {top_title}")
print(f"Hybridity score: {top_score:.4f}")
print()

act_heatmap = heatmap_ok[heatmap_ok['celex'] == top_celex][['articolo_id'] + pct_cols + ['entropy']].copy()
act_heatmap = act_heatmap.sort_values('entropy', ascending=False)

# Rinomina colonne pct__ → nome layer (abbreviato)
rename_map = {col: col_to_layer.get(col, col)[:25] for col in pct_cols}
act_display = act_heatmap.rename(columns=rename_map)

# Formatta percentuali senza decimali per leggibilità
layer_display_cols = [rename_map[c] for c in pct_cols]
for c in layer_display_cols:
    act_display[c] = act_display[c].apply(lambda x: f"{x:.0f}%")

act_display['entropia'] = act_heatmap['entropy'].apply(lambda x: f"{x:.3f}")

with pd.option_context('display.max_columns', None, 'display.width', 200,
                        'display.max_rows', 50):
    print(act_display[['articolo_id'] + layer_display_cols + ['entropia']].to_string(index=False))

In [ ]:
print("=" * 60)
print("DISTRIBUZIONE LAYER DOMINANTE")
print("=" * 60)

dominant_counts = hybridity_df['dominant_layer'].value_counts()
for layer, count in dominant_counts.items():
    pct = count / len(hybridity_df) * 100
    bar = '█' * int(pct)
    print(f"  {layer[:45]:.<46} {count:>4} atti  ({pct:4.1f}%)  {bar}")

print()
print("=" * 60)
print("ATTI PER PIPELINE LEVEL × LAYER DOMINANTE")
print("=" * 60)

if 'pipeline_level' in hybridity_df.columns:
    cross = pd.crosstab(
        hybridity_df['pipeline_level'],
        hybridity_df['dominant_layer']
    )
    print(cross.to_string())
else:
    print("Colonna pipeline_level non disponibile.")

In [ ]:
# Visualizzazione testuale: distribuzione score ibridità
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Istogramma ibridità
ax1 = axes[0]
ax1.hist(hybridity_df['hybridity_score'], bins=20, edgecolor='black', color='steelblue')
ax1.axvline(hybridity_df['hybridity_score'].median(), color='red',
            linestyle='--', label=f'Mediana = {hybridity_df["hybridity_score"].median():.3f}')
ax1.set_xlabel('Hybridity Score (entropia media)')
ax1.set_ylabel('N atti')
ax1.set_title('Distribuzione Score di Ibridità')
ax1.legend()

# Top 15 atti ibridi
ax2 = axes[1]
top15 = hybridity_df.head(15).copy()
labels = top15['celex'].apply(lambda x: str(x)[:14])
ax2.barh(range(len(top15)), top15['hybridity_score'], color='tomato')
ax2.set_yticks(range(len(top15)))
ax2.set_yticklabels(labels, fontsize=8)
ax2.invert_yaxis()
ax2.set_xlabel('Hybridity Score')
ax2.set_title('Top 15 Atti più Ibridi')

plt.tight_layout()

fig_path = os.path.join(output_path, 'figures', 'hybridity_distribution.png')
os.makedirs(os.path.dirname(fig_path), exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figura salvata: {fig_path}")

## 9. Riepilogo Output

Questa cella verifica che tutti i file di output siano stati prodotti correttamente.

In [ ]:
output_files = {
    'segments_descriptions.csv': SEGMENTS_DESC_FILE,
    'layer_mapping.csv':         LAYER_MAPPING_FILE,
    'nodes_heatmap.csv':         NODES_HEATMAP_FILE,
    'nodes_hybridity.csv':       NODES_HYBRIDITY_FILE,
}

print("=" * 60)
print("OUTPUT FILES")
print("=" * 60)

all_ok = True
for name, path in output_files.items():
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        df_tmp  = pd.read_csv(path)
        print(f"  ✓ {name}")
        print(f"    Righe: {len(df_tmp):,}  |  Dim: {size_kb:.1f} KB")
        print(f"    Colonne: {list(df_tmp.columns)[:6]}{'...' if len(df_tmp.columns) > 6 else ''}")
    else:
        print(f"  ✗ {name} — FILE MANCANTE")
        all_ok = False
    print()

if all_ok:
    print("✓ Tutti gli output presenti. Pipeline 04 completata.")
    print("  → Il notebook 05_network_analysis.ipynb può ora essere eseguito.")
else:
    print("⚠️  Alcuni file mancano — rieseguire le celle corrispondenti.")